### Importing Packages

In [4]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import nltk
import pandas as pd
import numpy as np

### Load Dataset

In [5]:
airline = pd.read_parquet('airline_clean.parquet', engine='pyarrow')
airline.dtypes

airline                       object
mentions                      object
hashtags                      object
tweet_created    datetime64[ns, UTC]
day                            int32
hour                           int32
day_of_week                    int32
day_name                      object
retweet_count                  int64
cleaned_text                  object
str_len                        int64
dtype: object

In [6]:
airline.head(1)

,airline,mentions,hashtags,tweet_created,day,hour,day_of_week,day_name,retweet_count,cleaned_text,str_len
0,Virgin America,"[@VirginAmerica, @dhepburn]",None,2015-02-24 19:35:52+00:00,24,19,1,Tuesday,0,what said.,10


In [7]:
airline['airline_num'] = airline['airline'].astype('category').cat.codes

In [8]:
airline['day_num'] = airline['day_of_week']
airline[['airline', 'airline_num', 'day_num']].drop_duplicates().head()

,airline,airline_num,day_num
0,Virgin America,5,1
49,Virgin America,5,0
129,Virgin America,5,6
195,Virgin America,5,5
270,Virgin America,5,4


In [9]:
airline['tweet_num'] = airline.groupby(['airline_num', 'day_num']).cumcount()
airline[['airline_num', 'day_num', 'tweet_num']].head()

,airline_num,day_num,tweet_num
0,5,1,0
1,5,1,1
2,5,1,2
3,5,1,3
4,5,1,4


In [10]:
LIB = (airline
    .groupby(['airline_num', 'airline'])
    .agg(
        total_tweets  = ('cleaned_text', 'count'),
        avg_length    = ('str_len', 'mean'),
        total_retweets= ('retweet_count', 'sum'),
        date_start    = ('tweet_created', 'min'),
        date_end      = ('tweet_created', 'max'),
    )
    .round(2)
    .reset_index()
    .set_index('airline_num')
    .sort_index()
)
print(LIB[['airline', 'avg_length']])

                    airline  avg_length
airline_num                            
0                  American      113.83
1                     Delta       98.33
2                 Southwest      105.45
3                US Airways      116.51
4                    United      114.16
5            Virgin America       98.28


In [11]:
LIB

,airline,total_tweets,avg_length,total_retweets,date_start,date_end
airline_num,,,,,,
0,American,2759,113.83,117,2015-02-19 04:25:30+00:00,2015-02-24 19:44:31+00:00
1,Delta,2222,98.33,252,2015-02-17 07:36:05+00:00,2015-02-24 19:48:38+00:00
2,Southwest,2420,105.45,145,2015-02-17 08:00:36+00:00,2015-02-24 19:47:53+00:00
3,US Airways,2913,116.51,249,2015-02-17 11:14:32+00:00,2015-02-24 19:53:37+00:00
4,United,3822,114.16,421,2015-02-17 07:48:48+00:00,2015-02-24 19:42:48+00:00
5,Virgin America,504,98.28,26,2015-02-17 16:43:42+00:00,2015-02-24 19:35:52+00:00


In [12]:
LIB.to_csv('LIB.csv')

### Tokenization

In [13]:
from nltk.tokenize import sent_tokenize

SENTS = (airline
    .set_index(['airline_num', 'day_num', 'tweet_num'])['cleaned_text']
    .apply(lambda x: sent_tokenize(str(x)))
    .explode()
    .to_frame('sent_str')
    .reset_index()
)
SENTS['sent_num'] = SENTS.groupby(['airline_num', 'day_num', 'tweet_num']).cumcount()
SENTS = SENTS.set_index(['airline_num', 'day_num', 'tweet_num', 'sent_num'])
SENTS.head()

sent_str
airline_num day_num tweet_num sent_num                                                   
5           1       0         0                                                what said.
                    1         0         plus you have added commercials time out the e...
                    2         0         i did not today.. must mean i need time out ta...
                    3         0         information technology is really aggressive ti...
                    4         0         and information technology is a really big bad...

In [14]:
TOKENS = (SENTS['sent_str']
    .apply(lambda x: word_tokenize(str(x)))
    .explode()
    .to_frame('token_str')
    .reset_index()
)

TOKENS['token_num'] = TOKENS.groupby(
    ['airline_num', 'day_num', 'tweet_num', 'sent_num']
).cumcount()

TOKENS = TOKENS.set_index(
    ['airline_num', 'day_num', 'tweet_num', 'sent_num', 'token_num']
)
TOKENS

token_str
airline_num day_num tweet_num sent_num token_num          
5           1       0         0        0              what
                                       1              said
                                       2                 .
                    1         0        0              plus
                                       1               you
...                                                    ...
0           6       403       1        9                on
                                       10              the
                                       11             next
                                       12           flight
                                       13                ?

[319752 rows x 1 columns]

### POS Tagging

In [15]:
from nltk.corpus import wordnet

def tag_sentence(group):
    tagged = nltk.pos_tag(group['token_str'].tolist())
    group = group.copy()
    group['pos'] = [tag for _, tag in tagged]
    return group

TOKENS = (TOKENS
    .reset_index()
    .groupby(['airline_num', 'day_num', 'tweet_num', 'sent_num'], group_keys=False)
    .apply(tag_sentence)
    .set_index(['airline_num', 'day_num', 'tweet_num', 'sent_num', 'token_num'])
)
TOKENS.head(10)

C:\Users\karina mehta\AppData\Local\Temp\ipykernel_3220\1925570065.py:12: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(tag_sentence)


token_str  pos
airline_num day_num tweet_num sent_num token_num                  
5           1       0         0        0                 what   WP
                                       1                 said  VBD
                                       2                    .    .
                    1         0        0                 plus   CC
                                       1                  you  PRP
                                       2                 have  VBP
                                       3                added  VBN
                                       4          commercials  NNS
                                       5                 time   NN
                                       6                  out   IN

### Lemmatization

In [16]:
def get_wordnet_pos(tag):
    if tag.startswith('J'): return wordnet.ADJ
    if tag.startswith('V'): return wordnet.VERB
    if tag.startswith('N'): return wordnet.NOUN
    if tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN

lemmatizer = WordNetLemmatizer()
TOKENS['lemma'] = TOKENS.apply(
    lambda row: lemmatizer.lemmatize(row['token_str'], get_wordnet_pos(row['pos'])),
    axis=1
)
TOKENS.head(10)

token_str  pos       lemma
airline_num day_num tweet_num sent_num token_num                              
5           1       0         0        0                 what   WP        what
                                       1                 said  VBD         say
                                       2                    .    .           .
                    1         0        0                 plus   CC        plus
                                       1                  you  PRP         you
                                       2                 have  VBP        have
                                       3                added  VBN         add
                                       4          commercials  NNS  commercial
                                       5                 time   NN        time
                                       6                  out   IN         out

In [17]:
TOKENS['term_str'] = TOKENS['lemma']

def get_pos_group(tag):
    if tag.startswith('NN'):                    return 'NOUN'
    if tag.startswith('VB') or tag == 'MD':     return 'VERB'
    if tag.startswith('JJ'):                    return 'ADJ'
    if tag.startswith('RB'):                    return 'ADV'
    if tag.startswith('PRP') or tag.startswith('WP'): return 'PRON'
    if tag in ('DT', 'WDT', 'PDT'):             return 'DET'
    if tag in ('IN', 'TO'):                     return 'PREP'
    if tag == 'CC':                             return 'CONJ'
    if tag == 'CD':                             return 'NUM'
    return 'OTHER'

TOKENS['pos_group'] = TOKENS['pos'].apply(get_pos_group)
TOKENS.head(10)

token_str  pos  \
airline_num day_num tweet_num sent_num token_num                     
5           1       0         0        0                 what   WP   
                                       1                 said  VBD   
                                       2                    .    .   
                    1         0        0                 plus   CC   
                                       1                  you  PRP   
                                       2                 have  VBP   
                                       3                added  VBN   
                                       4          commercials  NNS   
                                       5                 time   NN   
                                       6                  out   IN   

                                                       lemma    term_str  \
airline_num day_num tweet_num sent_num token_num                           
5           1       0         0        0                what        what   
                                       1                 say         say   
                                       2                   .           .   
                    1         0        0                plus        plus   
                                       1                 you         you   
                                       2                have        have   
                                       3                 add         add   
                                       4          commercial  commercial   
                                       5                time        time   
                                       6                 out         out   

                                                 pos_group  
airline_num day_num tweet_num sent_num token_num            
5           1       0         0        0              PRON  
                                       1              VERB  
                                       2             OTHER  
                    1         0        0              CONJ  
                                       1              PRON  
                                       2              VERB  
                                       3              VERB  
                                       4              NOUN  
                                       5              NOUN  
                                       6              PREP

In [18]:
import string, re

TOKENS = TOKENS[~TOKENS['token_str'].apply(lambda x: all(c in string.punctuation for c in str(x)))]

# Remove: pure numbers, colons from demojized emojis (:smile:), 
# single characters, and anything not purely alphabetic/hyphenated
TOKENS = TOKENS[TOKENS['token_str'].str.match(r'^[a-zA-Z][a-zA-Z\-]{1,}$')]

print(f"TOKENS after cleaning: {len(TOKENS):,}")
TOKENS.head()

TOKENS after cleaning: 262,745


token_str  pos lemma  \
airline_num day_num tweet_num sent_num token_num                        
5           1       0         0        0              what   WP  what   
                                       1              said  VBD   say   
                    1         0        0              plus   CC  plus   
                                       1               you  PRP   you   
                                       2              have  VBP  have   

                                                 term_str pos_group  
airline_num day_num tweet_num sent_num token_num                     
5           1       0         0        0             what      PRON  
                                       1              say      VERB  
                    1         0        0             plus      CONJ  
                                       1              you      PRON  
                                       2             have      VERB

### Remove stop words

In [19]:
stop_words = set(stopwords.words('english'))
TOKENS['is_stop'] = TOKENS['token_str'].isin(stop_words)
TOKENS.head(10)

token_str  pos  \
airline_num day_num tweet_num sent_num token_num                     
5           1       0         0        0                 what   WP   
                                       1                 said  VBD   
                    1         0        0                 plus   CC   
                                       1                  you  PRP   
                                       2                 have  VBP   
                                       3                added  VBN   
                                       4          commercials  NNS   
                                       5                 time   NN   
                                       6                  out   IN   
                                       7                  the   DT   

                                                       lemma    term_str  \
airline_num day_num tweet_num sent_num token_num                           
5           1       0         0        0                what        what   
                                       1                 say         say   
                    1         0        0                plus        plus   
                                       1                 you         you   
                                       2                have        have   
                                       3                 add         add   
                                       4          commercial  commercial   
                                       5                time        time   
                                       6                 out         out   
                                       7                 the         the   

                                                 pos_group  is_stop  
airline_num day_num tweet_num sent_num token_num                     
5           1       0         0        0              PRON     True  
                                       1              VERB    False  
                    1         0        0              CONJ    False  
                                       1              PRON     True  
                                       2              VERB     True  
                                       3              VERB    False  
                                       4              NOUN    False  
                                       5              NOUN    False  
                                       6              PREP     True  
                                       7               DET     True

In [20]:
print(f"Observations : {len(TOKENS)}")
print(f"OHCO: airline_num,day_num,tweet_num,sent_num,token_num")
print(f"Columns: {','.join(TOKENS.reset_index().columns.tolist())}")

TOKENS.to_csv('TOKENS.csv')

Observations : 262745
OHCO: airline_num,day_num,tweet_num,sent_num,token_num
Columns: airline_num,day_num,tweet_num,sent_num,token_num,token_str,pos,lemma,term_str,pos_group,is_stop


### VOCAB
The unique word types (terms) in the corpus.

In [21]:
from nltk.stem import PorterStemmer
import numpy as np

ps = PorterStemmer()
tok = TOKENS.reset_index()

# Core counts per term
VOCAB = (TOKENS
    .groupby('term_str')
    .agg(
        n             = ('term_str', 'count'),
        max_pos       = ('pos',       lambda x: x.value_counts().idxmax()),
        max_pos_group = ('pos_group', lambda x: x.value_counts().idxmax()),
        stop          = ('is_stop',   'max'),
    )
)

# Probability and self-information
N = VOCAB['n'].sum()
VOCAB['p'] = VOCAB['n'] / N
VOCAB['i'] = -np.log2(VOCAB['p'])

# Document frequency unique tweets containing each term
N_docs = tok[['airline_num','day_num','tweet_num']].drop_duplicates().shape[0]
df_series = (tok[['airline_num','day_num','tweet_num','term_str']]
    .drop_duplicates()
    .groupby('term_str')
    .size()
    .rename('df')
)
VOCAB = VOCAB.join(df_series)
VOCAB['idf']   = np.log(N_docs / VOCAB['df'])
VOCAB['dfidf'] = VOCAB['df'] * VOCAB['idf']

# Porter stem
VOCAB['porter_stem'] = VOCAB.index.map(ps.stem)

# Ngram length
VOCAB['ngram_length'] = VOCAB.index.map(lambda x: len(x.split()))

VOCAB = VOCAB.sort_values('dfidf', ascending=False)
print(f"VOCAB size: {len(VOCAB):,}")
VOCAB.head()

VOCAB size: 9,668


,n,max_pos,max_pos_group,stop,p,i,df,idf,dfidf,porter_stem,ngram_length
term_str,,,,,,,,,,,
the,7647,DT,DET,True,0.029104,5.102626,5760,0.932273,5369.894994,the,1
you,8476,PRP,PRON,True,0.032259,4.954136,6171,0.863350,5327.732923,you,1
be,8520,VBZ,VERB,True,0.032427,4.946666,6409,0.825508,5290.678584,be,1
and,5427,CC,CONJ,True,0.020655,5.597365,4368,1.208906,5280.499978,and,1
time,9164,NN,NOUN,False,0.034878,4.841542,6625,0.792361,5249.388572,time,1


In [22]:
# Top 20 significant words by DFIDF
VOCAB[~VOCAB['stop']].sort_values('dfidf', ascending=False).head(20)[
    ['n','p','i','df','dfidf','porter_stem','max_pos','max_pos_group','stop']
]

,n,p,i,df,dfidf,porter_stem,max_pos,max_pos_group,stop
term_str,,,,,,,,,
time,9164,0.034878,4.841542,6625,5249.388572,time,NN,NOUN,False
flight,4719,0.017960,5.799038,3887,5152.503259,flight,NN,NOUN,False
miss,3673,0.013979,6.160561,3051,4783.179333,miss,NN,NOUN,False
wait,2426,0.009233,6.758940,2110,4086.063453,wait,NN,NOUN,False
get,2117,0.008057,6.955498,1953,3933.037838,get,VB,VERB,False
information,2136,0.008130,6.942608,1898,3876.494580,inform,NN,NOUN,False
technology,2073,0.007890,6.985799,1844,3819.428861,technolog,NN,NOUN,False
see,2002,0.007620,7.036078,1774,3743.093924,see,VB,VERB,False
hell,1523,0.005796,7.430604,1450,3351.888664,hell,NN,NOUN,False


In [23]:
VOCAB.to_csv('VOCAB.csv')
print(f"Observations: {len(VOCAB)}")
print(f"Columns: {','.join(VOCAB.reset_index().columns.tolist())}")


Observations: 9668
Columns: term_str,n,max_pos,max_pos_group,stop,p,i,df,idf,dfidf,porter_stem,ngram_length



## Bag of Words, DTM & TF-IDF

In [24]:
MIN_FREQ     = 5                              # min corpus frequency to include
REMOVE_STOP  = True                           # drop stopwords
POS_FILTER   = ['NOUN', 'VERB', 'ADJ', 'ADV']# None = keep all POS groups
TOP_DFIDF    = None                           # int = top N by DFIDF, None = all
DOC_LEVEL    = ['airline_num','day_num','tweet_num'] 
N_COMPONENTS = 20                             # SVD components for reduced TFIDF

### BOW

In [25]:
SIG_VOCAB = VOCAB[
    (~VOCAB['stop']) &
    (VOCAB['max_pos_group'].isin(POS_FILTER)) &
    (VOCAB['n'] >= MIN_FREQ) &
    (VOCAB.index.str.match(r'^[a-z][a-z\-]+$'))   # alphabetic terms only, no punctuation/emoji artifacts
].copy()

print(f"SIG_VOCAB: {len(SIG_VOCAB):,} terms")
SIG_VOCAB.head()

SIG_VOCAB: 2,246 terms


,n,max_pos,max_pos_group,stop,p,i,df,idf,dfidf,porter_stem,ngram_length
term_str,,,,,,,,,,,
time,9164,NN,NOUN,False,0.034878,4.841542,6625,0.792361,5249.388572,time,1
flight,4719,NN,NOUN,False,0.017960,5.799038,3887,1.325573,5152.503259,flight,1
miss,3673,NN,NOUN,False,0.013979,6.160561,3051,1.567742,4783.179333,miss,1
wait,2426,NN,NOUN,False,0.009233,6.758940,2110,1.936523,4086.063453,wait,1
get,2117,VB,VERB,False,0.008057,6.955498,1953,2.013844,3933.037838,get,1


### DTM 

In [27]:
DTM = (tok[tok['term_str'].isin(SIG_VOCAB.index)]
    .groupby(DOC_LEVEL + ['term_str'])
    .size()
    .unstack('term_str')
    .fillna(0)
    .astype(int)
)
print(f"DTM shape: {DTM.shape}")
DTM.head()

DTM shape: (14556, 2246)


term_str                       a-list  aa  aadvantage  ability  able  abq  \
airline_num day_num tweet_num                                               
0           0       0               0   0           0        0     1    0   
                    1               0   0           0        0     0    0   
                    2               0   1           0        0     0    0   
                    3               0   0           0        0     0    0   
                    4               0   0           0        0     0    0   

term_str                       absolute  absolutely  absurd  abysmal  ...  \
airline_num day_num tweet_num                                         ...   
0           0       0                 0           0       0        0  ...   
                    1                 0           0       0        0  ...   
                    2                 0           0       0        0  ...   
                    3                 0           0       0        0  ...   
                    4                 0           0       0        0  ...   

term_str                       yesterday  yet  yo  york  young  yr  yup  yvr  \
airline_num day_num tweet_num                                                  
0           0       0                  0    0   0     0      0   0    0    0   
                    1                  0    0   0     0      0   0    0    0   
                    2                  0    0   0     0      0   0    0    0   
                    3                  0    0   0     0      0   0    0    0   
                    4                  0    0   0     0      0   0    0    0   

term_str                       yyz  zone  
airline_num day_num tweet_num             
0           0       0            0     0  
                    1            0     0  
                    2            0     0  
                    3            0     0  
                    4            0     0  

[5 rows x 2246 columns]

In [28]:
DTM.to_parquet('DTM.parquet')

### TF-IDF

In [29]:
TF    = DTM.div(DTM.sum(axis=1), axis=0)
IDF   = VOCAB.loc[SIG_VOCAB.index, 'idf']
TFIDF = TF.multiply(IDF, axis=1)

print(f"TF-IDF shape: {TFIDF.shape}")
TFIDF.head()

TF-IDF shape: (14556, 2246)


term_str                       a-list        aa  aadvantage  ability  \
airline_num day_num tweet_num                                          
0           0       0             0.0  0.000000         0.0      0.0   
                    1             0.0  0.000000         0.0      0.0   
                    2             0.0  0.360259         0.0      0.0   
                    3             0.0  0.000000         0.0      0.0   
                    4             0.0  0.000000         0.0      0.0   

term_str                           able  abq  absolute  absolutely  absurd  \
airline_num day_num tweet_num                                                
0           0       0          0.343105  0.0       0.0         0.0     0.0   
                    1          0.000000  0.0       0.0         0.0     0.0   
                    2          0.000000  0.0       0.0         0.0     0.0   
                    3          0.000000  0.0       0.0         0.0     0.0   
                    4          0.000000  0.0       0.0         0.0     0.0   

term_str                       abysmal  ...  yesterday  yet   yo  york  young  \
airline_num day_num tweet_num           ...                                     
0           0       0              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    1              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    2              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    3              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    4              0.0  ...        0.0  0.0  0.0   0.0    0.0   

term_str                        yr  yup  yvr  yyz  zone  
airline_num day_num tweet_num                            
0           0       0          0.0  0.0  0.0  0.0   0.0  
                    1          0.0  0.0  0.0  0.0   0.0  
                    2          0.0  0.0  0.0  0.0   0.0  
                    3          0.0  0.0  0.0  0.0   0.0  
                    4          0.0  0.0  0.0  0.0   0.0  

[5 rows x 2246 columns]

In [30]:
TFIDF.to_parquet('TFIDF.parquet')

In [31]:
# BOW: long-format sparse count table indexed by (bag + term_str)
BOW = (
    DTM
    .stack()
    .rename("n")
    .to_frame()
)
BOW = BOW[BOW["n"] > 0]

# Add tfidf values
tfidf_long = TFIDF.stack().rename("tfidf")
BOW["tfidf"] = tfidf_long

BOW.to_csv("BOW.csv")
print(f"BOW shape: {len(BOW):,} rows")
print(f"Index: {BOW.index.names}")
BOW.head()

BOW shape: 133,565 rows
Index: ['airline_num', 'day_num', 'tweet_num', 'term_str']


n     tfidf
airline_num day_num tweet_num term_str              
0           0       0         able       1  0.343105
                              author     1  0.196002
                              call       1  0.217956
                              cancelled  1  0.327644
                              drop       1  0.390274

### Normalized TF-IDF (L2)

In [32]:
from sklearn.preprocessing import normalize

TFIDF_NORM = pd.DataFrame(
    normalize(TFIDF.values, norm='l2'),
    index=TFIDF.index,
    columns=TFIDF.columns
)
print(f"Normalized TF-IDF shape: {TFIDF_NORM.shape}")
TFIDF_NORM.head()

Normalized TF-IDF shape: (14556, 2246)


term_str                       a-list        aa  aadvantage  ability  \
airline_num day_num tweet_num                                          
0           0       0             0.0  0.000000         0.0      0.0   
                    1             0.0  0.000000         0.0      0.0   
                    2             0.0  0.286214         0.0      0.0   
                    3             0.0  0.000000         0.0      0.0   
                    4             0.0  0.000000         0.0      0.0   

term_str                           able  abq  absolute  absolutely  absurd  \
airline_num day_num tweet_num                                                
0           0       0          0.335458  0.0       0.0         0.0     0.0   
                    1          0.000000  0.0       0.0         0.0     0.0   
                    2          0.000000  0.0       0.0         0.0     0.0   
                    3          0.000000  0.0       0.0         0.0     0.0   
                    4          0.000000  0.0       0.0         0.0     0.0   

term_str                       abysmal  ...  yesterday  yet   yo  york  young  \
airline_num day_num tweet_num           ...                                     
0           0       0              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    1              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    2              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    3              0.0  ...        0.0  0.0  0.0   0.0    0.0   
                    4              0.0  ...        0.0  0.0  0.0   0.0    0.0   

term_str                        yr  yup  yvr  yyz  zone  
airline_num day_num tweet_num                            
0           0       0          0.0  0.0  0.0  0.0   0.0  
                    1          0.0  0.0  0.0  0.0   0.0  
                    2          0.0  0.0  0.0  0.0   0.0  
                    3          0.0  0.0  0.0  0.0   0.0  
                    4          0.0  0.0  0.0  0.0   0.0  

[5 rows x 2246 columns]

In [33]:
TFIDF_NORM.to_parquet('TFIDF_NORM.parquet')

### Reduced TF-IDF (SVD / LSA)

In [34]:
from sklearn.decomposition import TruncatedSVD

svd = TruncatedSVD(n_components=N_COMPONENTS, random_state=42)

TFIDF_REDUCED = pd.DataFrame(
    svd.fit_transform(TFIDF_NORM),
    index=TFIDF_NORM.index,
    columns=[f'PC{i+1}' for i in range(N_COMPONENTS)]
)

print(f"Reduced TF-IDF shape: {TFIDF_REDUCED.shape}")
print(f"Variance explained: {svd.explained_variance_ratio_.sum():.2%}")
TFIDF_REDUCED.head()

Reduced TF-IDF shape: (14556, 20)
Variance explained: 12.54%


PC1       PC2       PC3       PC4  \
airline_num day_num tweet_num                                           
0           0       0          0.224326 -0.045270 -0.041571 -0.072785   
                    1          0.101145 -0.016451 -0.004030 -0.009839   
                    2          0.122804 -0.015778 -0.004756 -0.024897   
                    3          0.132349 -0.030653 -0.015344 -0.038086   
                    4          0.133784 -0.014435 -0.016078 -0.035059   

                                    PC5       PC6       PC7       PC8  \
airline_num day_num tweet_num                                           
0           0       0         -0.119252  0.022427  0.008694  0.181380   
                    1         -0.021091  0.025100  0.017014 -0.004592   
                    2         -0.036232  0.045283  0.034837  0.126471   
                    3         -0.017758  0.019189  0.013428  0.063117   
                    4         -0.056246 -0.011978  0.010808 -0.029823   

                                    PC9      PC10      PC11      PC12  \
airline_num day_num tweet_num                                           
0           0       0          0.072066 -0.181953  0.112265 -0.053429   
                    1         -0.044777 -0.002043 -0.009666  0.011527   
                    2         -0.057887  0.054721 -0.150947 -0.115300   
                    3          0.025829  0.080613 -0.126034 -0.018741   
                    4         -0.094900  0.018136 -0.036995  0.005383   

                                   PC13      PC14      PC15      PC16  \
airline_num day_num tweet_num                                           
0           0       0          0.090410  0.016342  0.020745 -0.054004   
                    1          0.061720  0.056304  0.010156 -0.035060   
                    2          0.021727 -0.011084  0.047669 -0.001415   
                    3          0.026351 -0.040795  0.006138  0.007300   
                    4         -0.038471  0.052618 -0.039933 -0.015104   

                                   PC17      PC18      PC19      PC20  
airline_num day_num tweet_num                                          
0           0       0         -0.000758  0.050201  0.012264 -0.043808  
                    1         -0.025905  0.059145  0.028801  0.017715  
                    2         -0.030635  0.034572 -0.044472 -0.076489  
                    3         -0.036056  0.024355 -0.058533 -0.051362  
                    4         -0.039128  0.066060  0.033093  0.070988

In [35]:
TFIDF_REDUCED.to_csv('TFIDF_REDUCED.csv')